# 步驟 2：因子工程與標籤建立

計算技術指標、動能、波動度、籌碼，以及未來一期的 Excess Return 超額報酬作為機器學習目標（Target）。

**14 個原始因子（後續會自動篩選 Top-8 ICIR）**

| 類別 | 因子 | 說明 |
|------|------|------|
| 動能 | `mom_1m`, `mom_3m`, `mom_6m` | 1/3/6 月報酬率 |
| 動能 | `mom_1m_ra` | 風險調整後動量（動量 / 20日波動）|
| 動能 | `price_52w` | 股價距 52 週高點比率 |
| 動能 | `breakout` | 超越 20 日高點比率 |
| 技術 | `rsi_14` | 14 日 RSI |
| 波動 | `atr_rel` | 14 日 ATR / 收盤價（相對波動率）|
| 量能 | `vol_ratio` | 20 日均量 / 60 日均量 |
| 籌碼 | `foreign_net_20d` | 外資 20 日淨買超 / 成交量 |
| 籌碼 | `trust_net_10d` | 投信 10 日淨買超 / 成交量 |
| 營收 | `rev_yoy`, `rev_mom` | 營收年增率 / 月增率 |
| 營收 | `rev_accel` | 營收月增加速度 |

**前置條件**：已執行 `01_Data_Pipeline.ipynb`，變數 `close`, `high_wide`, `low_wide`, `money_wide`, `rev_wide`, `foreign_net`, `trust_net` 存在記憶體中。

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from tqdm import tqdm

# ── 全域常數
TOP_FACTORS_K = 8   # 自動選 ICIR 絕對值最高的 N 個因子

def resample_monthly(df):
    return df.resample('ME').last()

def to_long(df, name):
    """Wide -> Long (Pandas 3.0 safe: unstack instead of stack)"""
    s = df.unstack().swaplevel().sort_index()
    s.name = name
    return s

def zscore_xs(df):
    """Cross-sectional z-score with winsorize +/-3 sigma"""
    df = df.replace([np.inf, -np.inf], np.nan)
    mu  = df.mean(axis=1)
    std = df.std(axis=1).replace(0, np.nan)
    return (df.sub(mu, axis=0).div(std, axis=0)).clip(-3, 3)

In [ ]:
# ── 動能因子
mom_1m = close / close.shift(21) - 1
mom_3m = close / close.shift(63) - 1
mom_6m = close / close.shift(126) - 1

rvol_20   = close.pct_change().rolling(20).std().replace(0, np.nan)
mom_1m_ra = (mom_1m / rvol_20).replace([np.inf, -np.inf], np.nan)

high_52w     = close.rolling(252).max()
price_to_52w = close / high_52w.replace(0, np.nan)

high_20d        = close.rolling(20).max()
breakout_signal = (close / high_20d.replace(0, np.nan) - 1)

# ── RSI
def calc_rsi(price_df, period=14):
    delta = price_df.diff()
    gain  = delta.clip(lower=0).ewm(com=period-1, min_periods=period).mean()
    loss  = (-delta).clip(lower=0).ewm(com=period-1, min_periods=period).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

rsi_14 = calc_rsi(close)

# ── ATR
hl     = (high_wide - low_wide).abs()
atr_14 = hl.rolling(14).mean()
atr_rel = (atr_14 / close.replace(0, np.nan))

# ── 量能
vol_ratio = money_wide.rolling(20).mean() / money_wide.rolling(60).mean()

# ── 營收
rev_yoy   = rev_wide.pct_change(12)
rev_mom   = rev_wide.pct_change(1)
rev_accel = rev_wide.pct_change(1) - rev_wide.pct_change(1).shift(3)

rev_yoy_d   = rev_yoy.reindex(close.index, method='ffill')
rev_mom_d   = rev_mom.reindex(close.index, method='ffill')
rev_accel_d = rev_accel.reindex(close.index, method='ffill')

# ── 籌碼
vol_sum20       = money_wide.rolling(20).sum()
common_f        = foreign_net.columns.intersection(vol_sum20.columns)
common_t        = trust_net.columns.intersection(vol_sum20.columns)
foreign_net_20d = foreign_net[common_f].rolling(20).sum() / vol_sum20[common_f].replace(0, np.nan)
trust_net_10d   = trust_net[common_t].rolling(10).sum()  / vol_sum20[common_t].replace(0, np.nan)

print('所有因子計算完成')

In [ ]:
# ── 月頻率 + Z-score 標準化
factor_defs = {
    'mom_1m'         : resample_monthly(mom_1m),
    'mom_3m'         : resample_monthly(mom_3m),
    'mom_6m'         : resample_monthly(mom_6m),
    'rsi_14'         : resample_monthly(rsi_14),
    'vol_ratio'      : resample_monthly(vol_ratio),
    'rev_yoy'        : resample_monthly(rev_yoy_d),
    'rev_mom'        : resample_monthly(rev_mom_d),
    'foreign_net_20d': resample_monthly(foreign_net_20d),
    'trust_net_10d'  : resample_monthly(trust_net_10d),
    'mom_1m_ra'      : resample_monthly(mom_1m_ra),
    'price_52w'      : resample_monthly(price_to_52w),
    'breakout'       : resample_monthly(breakout_signal),
    'atr_rel'        : resample_monthly(atr_rel),
    'rev_accel'      : resample_monthly(rev_accel_d),
}

factors_z    = {name: zscore_xs(df) for name, df in factor_defs.items()}
combined_long = [to_long(df, name) for name, df in factors_z.items()]
X_raw = pd.concat(combined_long, axis=1)

print(f'X_raw shape: {X_raw.shape}')

In [ ]:
# ── 目標變數：下個月超額報酬
close_m     = resample_monthly(close)
monthly_ret = close_m.pct_change(1).shift(-1)
mkt_med     = monthly_ret.median(axis=1)
excess_ret  = monthly_ret.subtract(mkt_med, axis=0)
y_raw       = to_long(excess_ret, 'excess_return')

X_raw, y_raw = X_raw.align(y_raw, join='inner', axis=0)
X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
y_raw = y_raw.replace([np.inf, -np.inf], np.nan)
mask  = X_raw.notna().all(axis=1) & y_raw.notna()
X, y  = X_raw[mask], y_raw[mask]

all_dates = X.index.get_level_values(0).unique().sort_values()
print(f'Dataset: X={X.shape}, y={y.shape}')
print(f'Date range: {all_dates[0].date()} ~ {all_dates[-1].date()}')
print(f'Unique stocks: {X.index.get_level_values(1).nunique()}')

In [ ]:
# ── IC 分析：計算每個因子的 ICIR
# 注意：這裡是在全資料上計算 ICIR，屬於探索性分析，
# 生產環境理想上應在每次 walk-forward 訓練折內計算。
ic_results = {}
for factor in X.columns:
    ics = []
    for d in all_dates:
        try:
            x_d = X.loc[d, factor].dropna()
            y_d = y.loc[d].reindex(x_d.index).dropna()
            x_d = x_d.reindex(y_d.index)
            if len(y_d) > 10:
                ic, _ = spearmanr(x_d, y_d)
                if not np.isnan(ic):
                    ics.append(ic)
        except Exception:
            pass
    s = pd.Series(ics)
    ic_results[factor] = {'IC Mean': s.mean(), 'IC Std': s.std(),
                           'ICIR': s.mean()/(s.std()+1e-8), 'IC>0%': (s>0).mean()}

ic_df = (pd.DataFrame(ic_results).T
           .sort_values('ICIR', key=abs, ascending=False))
display(ic_df.round(4))

# ── 自動選出 Top-K 因子
top_factors = ic_df.head(TOP_FACTORS_K).index.tolist()
print(f'\n>> Auto-selected Top-{TOP_FACTORS_K} factors by |ICIR|: {top_factors}')
X = X[top_factors]
print(f'X (filtered) shape: {X.shape}')